# snapshot-v2 — последовательные абляции в Google Colab

Новый notebook относится только к `snapshot_v2`. Старый `colab_training.ipynb` остаётся неизменяемым сценарием `v1_baseline`.

**Не нажимай Run all.** Длительные ячейки запускаются строго по одной. После каждого кандидата решение принимается только по validation. Test и `snapshot_final_holdout` здесь не открываются: все runtime-конфигурации принудительно получают `evaluate_test: false`.

## 1. Drive, GPU и пути
Эти ячейки повторяются после каждого нового runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/diffusion-sources')
DATA_DIR = DRIVE_ROOT / 'data/facebook_main'
CACHE_PATH = DRIVE_ROOT / 'data/cache/ego_facebook_shortest_paths.npz'
RUN_ROOT = DRIVE_ROOT / 'reports/runs/facebook_main_v2'
STATE_PATH = RUN_ROOT / 'colab_ablation_state.json'
REPOSITORY = 'https://github.com/1habibi/diffusion-sources-localization-.git'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Выбери Runtime -> Change runtime type -> GPU'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Получение кода и установка
Все изменения S0–S7 должны быть заранее запушены. Notebook не удаляет существующий checkout.

In [ ]:
print('[setup 1/5] Синхронизация репозитория', flush=True)
REPO = Path('/content/diffusion-sources')
if not REPO.exists():
    !git clone {REPOSITORY} {REPO}
else:
    %cd {REPO}
    !git pull --ff-only
%cd {REPO}
print('[setup 2/5] Установка зависимостей', flush=True)
!pip install -q networkx numpy scipy matplotlib PyYAML scikit-learn streamlit tqdm torch-geometric
print('[setup 3/5] Установка проекта', flush=True)
!pip install -q -e . --no-deps
print('[setup 4/5] Проверка версии и компиляция', flush=True)
!git rev-parse HEAD
!python -m compileall -q src scripts
print('[setup 5/5] Запуск тестов', flush=True)
!pytest -q
print('[setup] Готово', flush=True)

## 3. Проверка development-данных
Проверяются только `graph/train/validation`. Наличие test и holdout намеренно не проверяется и их содержимое не читается.

In [ ]:
required = ['graph.npz', 'train.npz', 'validation.npz']
missing = [name for name in required if not (DATA_DIR / name).exists()]
assert not missing, f'Загрузи data/generated/facebook_main в {DATA_DIR}; не найдены {missing}'
for name in required:
    path = DATA_DIR / name
    print(name, round(path.stat().st_size / 1024**2, 2), 'MiB')
print('Distance cache:', CACHE_PATH, 'exists =', CACHE_PATH.exists())

## 4. Оркестратор изолированных шагов
Состояние текущего validation-победителя хранится на Drive. Каждый следующий конфиг строится от фактического победителя; отклонённый компонент автоматически не переносится дальше.

In [ ]:
import copy, hashlib, json, subprocess, time
import yaml

BASE = ['observed_infected', 'log_degree_normalized']
LOCAL = ['observed_neighbor_count_normalized', 'observed_neighbor_fraction', 'unobserved_neighbor_count_normalized', 'unobserved_neighbor_fraction', 'closed_neighborhood_observed_fraction']
DISTANCE = ['mean_distance_to_observed_normalized', 'max_distance_to_observed_normalized', 'induced_observed_eccentricity_normalized']
GLOBALS = ['observed_count_normalized', 'observed_subgraph_density', 'observed_component_count_normalized', 'observed_largest_component_fraction']
JORDAN = ['multi_jordan_rank_normalized']

def load_yaml(path):
    return yaml.safe_load(Path(path).read_text())

def save_state(state):
    STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    STATE_PATH.write_text(json.dumps(state, indent=2, ensure_ascii=False))

STATE = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {'current_config': None, 'current_output': None, 'accepted': [], 'runs': {}, 'decisions': []}

def _features(config):
    names = config['data'].get('feature_names')
    return list(names) if names is not None else list(BASE)

def _add(names, group):
    return names + [name for name in group if name not in names]

def build_config(label, step, *, seed=7026, lambda_rank=None, rank_negatives=8, hard_fraction=0.5):
    if step == 'S0':
        config = load_yaml(REPO / 'configs/snapshot_v2/s0_v1_joint.yaml')
    else:
        assert STATE['current_config'], 'Сначала выполни и зафиксируй S0'
        config = copy.deepcopy(load_yaml(STATE['current_config']))
        names = _features(config)
        config['data'].pop('feature_indices', None)
        config['data']['feature_names'] = names
        if step == 'S1a': config['data']['feature_names'] = _add(names, LOCAL)
        elif step == 'S1b': config['data']['feature_names'] = _add(names, DISTANCE)
        elif step == 'S1c': config['data']['feature_names'] = _add(names, GLOBALS)
        elif step == 'S2': config['model']['source_head_mode'] = 'global_context'
        elif step == 'S3': config['model']['backbone_mode'] = 'residual_3'
        elif step == 'S4': config['model']['source_head_strategy'] = 'specialized_k'
        elif step == 'S5':
            config['loss']['lambda_rank'] = float(lambda_rank)
            config['loss']['rank_negatives_per_positive'] = int(rank_negatives)
            config['loss']['rank_hard_negative_fraction'] = float(hard_fraction)
        elif step == 'S6': config['data']['feature_names'] = _add(names, JORDAN)
        elif step == 'S7':
            config['model']['shortlist_mode'] = 'preliminary'
            config['loss']['lambda_preliminary'] = 1.0
            config['shortlist'] = {'enabled': True, 'sizes': [8, 12, 16, 24, 32], 'bootstrap_repeats': 2000, 'bootstrap_seed': 17026, 'micro_candidate_recall_min': 0.95, 'per_k_candidate_recall_min': 0.95, 'bootstrap_ci_low_min': 0.93, 'require_f1_or_latency_improvement': True, 'fallback': 'full_candidate_scoring'}
        else: raise ValueError(step)

    names = config['data'].get('feature_names')
    config['data']['directory'] = str(DATA_DIR)
    if names and any(name in names for name in DISTANCE + JORDAN):
        config['data']['distance_cache'] = str(CACHE_PATH)
        config['data']['distance_cap'] = 10
    config['model'].setdefault('backbone_mode', 'plain_2')
    config['model'].setdefault('source_head_mode', 'local')
    config['model'].setdefault('source_head_strategy', 'shared')
    config['model'].setdefault('shortlist_mode', 'disabled')
    config['model']['input_dim'] = len(names) if names else len(config['data'].get('feature_indices', [0, 1]))
    config['model']['global_feature_dim'] = len([name for name in (names or []) if name in GLOBALS]) if config['model']['source_head_mode'] == 'global_context' else 0
    config['training'].update({'seed': int(seed), 'device': 'cuda', 'resume': True})
    config['loss'].setdefault('lambda_rank', 0.0)
    config['loss'].setdefault('lambda_preliminary', 0.0)
    config['evaluation'] = {'selection_split': 'validation', 'evaluate_test': False, 'test_policy': 'locked_until_candidate_freeze', 'final_holdout_policy': 'sealed_until_final_freeze'}
    config['experiment'] = label.lower()
    config['ablation'] = {'id': label, 'parent': STATE.get('current_output') or 'v1_baseline', 'changed_factor': step}
    return config

def run_training_job(config_path, output, label):
    command = ['python', '-u', 'scripts/train_model.py', '--config', str(config_path), '--output', str(output)]
    started = time.monotonic()
    print(f'[{label}] Старт. Прогресс ниже; checkpoint: {output / "last_checkpoint.pt"}', flush=True)
    try:
        subprocess.run(command, check=True)
    except KeyboardInterrupt:
        print(f'[{label}] Остановлено пользователем; следующий запуск продолжится с checkpoint, если он уже создан.', flush=True)
        raise
    elapsed = time.monotonic() - started
    print(f'[{label}] Готово за {elapsed / 60:.1f} мин.', flush=True)

def run_candidate(label, step, **kwargs):
    config = build_config(label, step, **kwargs)
    seed = config['training']['seed']
    output = RUN_ROOT / label.lower() / f'seed_{seed}'
    runtime_config = Path('/content') / f'{label.lower()}_{seed}.yaml'
    runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
    checksum = hashlib.sha256(runtime_config.read_bytes()).hexdigest()
    print('Config:', runtime_config, 'sha256:', checksum, flush=True)
    print('Output:', output, flush=True)
    run_training_job(runtime_config, output, label)
    metrics = json.loads((output / 'metrics.json').read_text())
    assert metrics['test_evaluated'] is False
    assert not (output / 'test_predictions.csv').exists()
    assert (output / 'validation_predictions.csv').exists()
    STATE['runs'][label] = str(output)
    save_state(STATE)
    show_run(label)
    return output

def validation_row(label):
    output = Path(STATE['runs'][label])
    metrics = json.loads((output / 'metrics.json').read_text())
    detail = metrics['validation_prediction_metrics']['joint_estimated_k']
    overall = detail['all']
    return {'label': label, 'f1': overall['f1'], 'k1_f1': detail.get('1', {}).get('f1'), 'count_accuracy': overall['count_accuracy'], 'exact': overall['exact_set_accuracy'], 'distance': overall['symmetric_set_distance'], 'hit1': overall['hit_at_1_hop'], 'parameters': metrics['parameter_count'], 'output': str(output)}

def show_run(label):
    row = validation_row(label)
    print(json.dumps(row, indent=2, ensure_ascii=False))
    metrics = json.loads((Path(row['output']) / 'metrics.json').read_text())
    if metrics.get('shortlist_validation'):
        print('Shortlist:', json.dumps(metrics['shortlist_validation'], indent=2))

def compare(current_label, candidate_label):
    rows = [validation_row(current_label), validation_row(candidate_label)]
    for row in rows: print(json.dumps(row, indent=2, ensure_ascii=False))
    print('Delta F1 =', rows[1]['f1'] - rows[0]['f1'], '; если |delta| < 0.005, выбирается более простая модель')

def decide(label, accept, note):
    assert isinstance(accept, bool) and note.strip()
    output = Path(STATE['runs'][label])
    STATE['decisions'].append({'label': label, 'accepted': accept, 'note': note, 'time': time.time()})
    if accept:
        STATE['current_output'] = str(output)
        STATE['current_config'] = str(output / 'config.yaml')
        STATE['accepted'].append(label)
    save_state(STATE)
    print('Current winner:', STATE['current_output'])

print(json.dumps(STATE, indent=2, ensure_ascii=False))

## 5. GPU smoke перед длинной серией
Проверяется финальный локальный pipeline на малых split. Результат пишется во временный `/content` и не участвует в выборе.

In [ ]:
smoke = load_yaml(REPO / 'configs/snapshot_v2/s7_safe_shortlist_smoke.yaml')
smoke['data']['directory'] = str(DATA_DIR)
smoke['data']['distance_cache'] = str(CACHE_PATH)
smoke['training']['device'] = 'cuda'
smoke['evaluation']['evaluate_test'] = False
smoke_path = Path('/content/snapshot_v2_gpu_smoke.yaml')
smoke_path.write_text(yaml.safe_dump(smoke, sort_keys=False))
run_training_job(smoke_path, Path('/content/snapshot_v2_gpu_smoke'), 'GPU smoke')
smoke_metrics = json.loads(Path('/content/snapshot_v2_gpu_smoke/metrics.json').read_text())
assert smoke_metrics['test_evaluated'] is False
assert not Path('/content/snapshot_v2_gpu_smoke/test_predictions.csv').exists()
print('GPU smoke готов; эти метрики не интерпретировать')

# 6. Последовательность S0–S4
Для каждого шага: запусти длинную ячейку, сравни с текущим победителем, затем отредактируй `ACCEPT` и комментарий в следующей ячейке. Не переходи дальше с `None`.

In [ ]:
run_candidate('S0', 'S0')
decide('S0', True, 'Контрольный baseline snapshot-v2; test закрыт')

In [ ]:
CURRENT = 'S0' if STATE['current_output'].endswith('/s0/seed_7026') else STATE['accepted'][-1]
run_candidate('S1a', 'S1a')
compare(CURRENT, 'S1a')

In [ ]:
ACCEPT_S1A = None  # замени на True/False после validation-сравнения
assert ACCEPT_S1A is not None
decide('S1a', ACCEPT_S1A, 'Запиши F1/delta и причину решения')

In [ ]:
CURRENT = STATE['accepted'][-1]
run_candidate('S1b', 'S1b')
compare(CURRENT, 'S1b')

In [ ]:
ACCEPT_S1B = None
assert ACCEPT_S1B is not None
decide('S1b', ACCEPT_S1B, 'Запиши F1/delta и причину решения')

In [ ]:
CURRENT = STATE['accepted'][-1]
run_candidate('S1c', 'S1c')
compare(CURRENT, 'S1c')

In [ ]:
ACCEPT_S1C = None
assert ACCEPT_S1C is not None
decide('S1c', ACCEPT_S1C, 'Запиши F1/delta и причину решения')

In [ ]:
CURRENT = STATE['accepted'][-1]
run_candidate('S2', 'S2')
compare(CURRENT, 'S2')

In [ ]:
ACCEPT_S2 = None
assert ACCEPT_S2 is not None
decide('S2', ACCEPT_S2, 'Запиши F1/delta и причину решения')

In [ ]:
CURRENT = STATE['accepted'][-1]
run_candidate('S3', 'S3')
compare(CURRENT, 'S3')

In [ ]:
ACCEPT_S3 = None
assert ACCEPT_S3 is not None
decide('S3', ACCEPT_S3, 'Запиши F1/delta, параметры/время и причину')

In [ ]:
CURRENT = STATE['accepted'][-1]
run_candidate('S4', 'S4')
compare(CURRENT, 'S4')

In [ ]:
ACCEPT_S4 = None
assert ACCEPT_S4 is not None
decide('S4', ACCEPT_S4, 'Обязательно укажи F1 по k=1/2/3 и отсутствие деградации')

# 7. S5 ranking-loss
Текущий победитель с `lambda_rank=0` является обязательным контролем. Ниже четыре ограниченные validation-точки; каждая запускается отдельно от одного и того же parent. После таблицы можно принять ровно одну или отклонить S5 целиком.

In [ ]:
S5_PARENT = STATE['accepted'][-1]
run_candidate('S5_L01_N8_H05', 'S5', lambda_rank=0.1, rank_negatives=8, hard_fraction=0.5)
compare(S5_PARENT, 'S5_L01_N8_H05')

In [ ]:
S5_PARENT = STATE['accepted'][-1]
run_candidate('S5_L02_N8_H05', 'S5', lambda_rank=0.2, rank_negatives=8, hard_fraction=0.5)
compare(S5_PARENT, 'S5_L02_N8_H05')

In [ ]:
S5_PARENT = STATE['accepted'][-1]
run_candidate('S5_L02_N4_H05', 'S5', lambda_rank=0.2, rank_negatives=4, hard_fraction=0.5)
compare(S5_PARENT, 'S5_L02_N4_H05')

In [ ]:
S5_PARENT = STATE['accepted'][-1]
run_candidate('S5_L02_N8_H075', 'S5', lambda_rank=0.2, rank_negatives=8, hard_fraction=0.75)
compare(S5_PARENT, 'S5_L02_N8_H075')

In [ ]:
S5_PARENT = STATE['accepted'][-1]
S5_LABELS = ['S5_L01_N8_H05', 'S5_L02_N8_H05', 'S5_L02_N4_H05', 'S5_L02_N8_H075']
print(json.dumps(validation_row(S5_PARENT), indent=2))
for label in S5_LABELS: print(json.dumps(validation_row(label), indent=2))
SELECTED_S5 = None  # None = отклонить; иначе одна строка из S5_LABELS
assert SELECTED_S5 is None or SELECTED_S5 in S5_LABELS
if SELECTED_S5 is not None: decide(SELECTED_S5, True, 'Выбран по validation из ограниченной S5-сетки')
else: print('S5 отклонён; текущий победитель не изменён')

# 8. S6 и S7
S6 добавляет только Multi-Jordan rank. S7 обучает detached preliminary-head и внутри одного checkpoint сравнивает полный scoring с сеткой shortlist.

In [ ]:
CURRENT = STATE['accepted'][-1]
run_candidate('S6', 'S6')
compare(CURRENT, 'S6')

In [ ]:
ACCEPT_S6 = None
assert ACCEPT_S6 is not None
decide('S6', ACCEPT_S6, 'Запиши validation F1/distance/Hit и причину')

In [ ]:
CURRENT = STATE['accepted'][-1]
run_candidate('S7', 'S7')
compare(CURRENT, 'S7')
s7 = json.loads((Path(STATE['runs']['S7']) / 'shortlist_validation.json').read_text())
print('Автоматическое решение:', s7['decision'], 'M =', s7['selected_size'])

In [ ]:
ACCEPT_S7 = None  # True разрешено только при s7['decision'] == 'eligible'
assert ACCEPT_S7 is not None
assert not ACCEPT_S7 or s7['decision'] == 'eligible'
decide('S7', ACCEPT_S7, f"shortlist decision={s7['decision']}, selected M={s7['selected_size']}")

# 9. Leave-one-component-out
Для каждого принятого компонента S1a–S6 нужно отдельно построить и обучить вариант без него. S7 уже имеет внутренний full-scoring control и повторного обучения не требует. Запускай следующую ячейку по одному значению `LOO_COMPONENT`; если удаление улучшает validation, компонент следует убрать и повторно заморозить конфигурацию.

In [ ]:
def build_loo_config(component):
    config = copy.deepcopy(load_yaml(STATE['current_config']))
    names = _features(config)
    if component == 'S1a': names = [x for x in names if x not in LOCAL]
    elif component == 'S1b': names = [x for x in names if x not in DISTANCE]
    elif component == 'S1c': names = [x for x in names if x not in GLOBALS]
    elif component == 'S2': config['model']['source_head_mode'] = 'local'
    elif component == 'S3': config['model']['backbone_mode'] = 'plain_2'
    elif component == 'S4': config['model']['source_head_strategy'] = 'shared'
    elif component == 'S5': config['loss']['lambda_rank'] = 0.0
    elif component == 'S6': names = [x for x in names if x not in JORDAN]
    else: raise ValueError(component)
    config['data']['feature_names'] = names
    config['model']['input_dim'] = len(names)
    config['model']['global_feature_dim'] = len([x for x in names if x in GLOBALS]) if config['model']['source_head_mode'] == 'global_context' else 0
    config.pop('shortlist', None)
    config['model']['shortlist_mode'] = 'disabled'
    config['loss']['lambda_preliminary'] = 0.0
    return config

LOO_COMPONENT = None  # например 'S3'; запускай по одному принятому компоненту
assert LOO_COMPONENT in {'S1a','S1b','S1c','S2','S3','S4','S5','S6'}
loo_label = f'LOO_{LOO_COMPONENT}'
loo = build_loo_config(LOO_COMPONENT)
loo_path = Path('/content') / f'{loo_label.lower()}.yaml'
loo_out = RUN_ROOT / 'leave_one_out' / loo_label.lower() / 'seed_7026'
loo_path.write_text(yaml.safe_dump(loo, sort_keys=False))
run_training_job(loo_path, loo_out, loo_label)
STATE['runs'][loo_label] = str(loo_out); save_state(STATE)
compare(STATE['accepted'][-1], loo_label)

# 10. Freeze и финальные training seeds
Выполняй только после решений S0–S7 и leave-one-out. Test и final holdout всё ещё закрыты. Сначала зафиксируй config checksum, затем обучи seed 7027 и 7028 по отдельности.

In [ ]:
FROZEN = False  # поставь True только после переноса всех validation-решений в журналы проекта
assert FROZEN
frozen_config = Path(STATE['current_config'])
freeze_manifest = {'config': str(frozen_config), 'config_sha256': hashlib.sha256(frozen_config.read_bytes()).hexdigest(), 'accepted': STATE['accepted'], 'decisions': STATE['decisions'], 'test_opened': False, 'final_holdout_opened': False}
freeze_path = RUN_ROOT / 'snapshot_v2_freeze_manifest.json'
freeze_path.write_text(json.dumps(freeze_manifest, indent=2, ensure_ascii=False))
print(json.dumps(freeze_manifest, indent=2, ensure_ascii=False))

In [ ]:
def run_frozen_seed(seed):
    config = copy.deepcopy(load_yaml(STATE['current_config']))
    config['training']['seed'] = int(seed)
    config['evaluation']['evaluate_test'] = False
    path = Path('/content') / f'snapshot_v2_frozen_{seed}.yaml'
    output = RUN_ROOT / 'frozen_candidate' / f'seed_{seed}'
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    run_training_job(path, output, f'Frozen seed {seed}')
    metrics = json.loads((output / 'metrics.json').read_text())
    assert metrics['test_evaluated'] is False and not (output / 'test_predictions.csv').exists()
    print(seed, metrics['validation_prediction_metrics']['joint_estimated_k'])
    return output

FROZEN_7027 = run_frozen_seed(7027)

In [ ]:
FROZEN_7028 = run_frozen_seed(7028)

# Стоп перед test/final holdout

Скачай/синхронизируй `metrics.json`, `validation_predictions.csv`, histories, checkpoints, runtime configs, `shortlist_validation.json` и freeze manifest. Обнови `development_log.md` и `docs/experiment_registry.md`. Только после проверки трёх seed, checksums и отдельного финального evaluator можно один раз открыть существующий test и запечатанный final holdout. Не меняй `evaluate_test` в этом notebook вручную.